# A1: InternVL2.5-2B + QLoRA

Вторая архитектура проекта. Здесь проверяется, как русскоязычные инструкции VK влияют на `OpenGVLab/InternVL2_5-2B` — модель с визуальным энкодером InternViT и языковой частью InternLM. Это отдельный эксперимент, поэтому его метрики будут сравниваться с B0 и F1-fast только после одинаковой оценки на GQA-ru и MMBench-ru.

**Конфигурация A1:** те же 2 000 примеров `LLaVA-Instruct-ru`, одна эпоха, QLoRA rank=8, batch size=1 и накопление градиента=8. Ориентир для RTX 4060 — около 2 часов. Длительное обучение в этом ноутбуке намеренно не запускается автоматически.

In [ ]:
import json
import os
from pathlib import Path

import pandas as pd
import torch
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms
from transformers import AutoModel, AutoTokenizer, BitsAndBytesConfig, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

os.environ['HF_HUB_DISABLE_XET'] = '1'
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW_DATA = ROOT / 'data' / 'raw' / 'llava_instruct_ru_train.json'
SUBSET_PATH = ROOT / 'data' / 'processed' / 'llava_instruct_ru_first_iteration_seed42.csv'
IMAGE_CACHE = ROOT / 'data' / 'coco_cache' / 'train2017'
MODEL_DIR = ROOT / 'models' / 'a1_internvl25_2b_qlora_adapter'
CHECKPOINT_DIR = ROOT / 'checkpoints' / 'a1_internvl25_2b_qlora'
RESULTS_DIR = ROOT / 'results'
for directory in (MODEL_DIR, CHECKPOINT_DIR, RESULTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

MODEL_ID = 'OpenGVLab/InternVL2_5-2B'
SEED = 42
TRAINING_EXAMPLES = 2_000
IMAGE_SIZE = 448
MAX_LENGTH = 1024
assert torch.cuda.is_available(), 'Для A1 нужна CUDA.'
print('GPU:', torch.cuda.get_device_name(0))

## 1. Та же фиксированная выборка VK

Чтобы сравнение архитектур было честным, A1 использует те же 2 000 строк, что и F1-fast. Изображения не скачиваются из сети: они уже лежат в локальном COCO-кэше.

In [ ]:
subset = pd.read_csv(SUBSET_PATH)
raw_records = json.loads(RAW_DATA.read_text(encoding='utf-8'))

def parse_record(record):
    messages = record['conversations']
    question = next(item['value'] for item in messages if item['from'] == 'human')
    answer = next(item['value'] for item in messages if item['from'] == 'gpt')
    return question.replace('<image>\n', '').strip(), answer.strip(), record['image']

rows = []
for row in subset[['row_number', 'type']].itertuples(index=False):
    question, answer, image_path = parse_record(raw_records[int(row.row_number)])
    rows.append({'type': row.type, 'question': question, 'answer': answer, 'image_path': image_path})
full_train_df = pd.DataFrame(rows)
train_df = (full_train_df.groupby('type', group_keys=False)
            .apply(lambda group: group.sample(n=round(TRAINING_EXAMPLES * len(group) / len(full_train_df)), random_state=SEED))
            .reset_index(drop=True))
train_df = train_df.sample(n=TRAINING_EXAMPLES, random_state=SEED).reset_index(drop=True)
missing = sum(not (IMAGE_CACHE / Path(path).name).exists() for path in train_df.image_path)
assert len(train_df) == TRAINING_EXAMPLES
print(f'Строк для A1: {len(train_df)}; отсутствующих изображений в кэше: {missing}')
display(train_df[['type', 'question', 'answer', 'image_path']].head(3))

## 2. Датасет и обработка изображений

InternVL принимает нормализованные тензоры 448×448. Для первой сравнимой итерации берём по одному центральному изображению без динамического разбиения на тайлы: так потребление памяти остаётся предсказуемым.

In [ ]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
image_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE), interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

def load_image(image_path):
    local_path = IMAGE_CACHE / Path(image_path).name
    if not local_path.exists():
        raise FileNotFoundError(f'Нет изображения в локальном кэше: {local_path}')
    return Image.open(local_path).convert('RGB')

class RussianInternVLDataset(Dataset):
    def __init__(self, frame):
        self.frame = frame.reset_index(drop=True)
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, index):
        row = self.frame.iloc[index]
        return {'question': row.question, 'answer': row.answer, 'image_path': row.image_path}

dataset = RussianInternVLDataset(train_df)
example = dataset[0]
print('Вопрос:', example['question'])
print('Ответ:', example['answer'][:180] + '...')
display(load_image(example['image_path']))

## 3. Загрузка модели и QLoRA

Модель загружается в 4-битном NF4-формате. Обучаются только LoRA-слои внимания и MLP языковой части; InternViT остаётся замороженным. Запускайте эту ячейку, только когда F1 уже завершён и GPU свободен.

In [ ]:
quantization = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True, use_fast=False)
model = AutoModel.from_pretrained(
    MODEL_ID, trust_remote_code=True, low_cpu_mem_usage=True,
    quantization_config=quantization, device_map='auto', torch_dtype=torch.float16,
).eval()
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
lora_config = LoraConfig(
    r=8, lora_alpha=16, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 4. Collator и короткая проверка forward pass

Перед длительным запуском создаём один обучающий батч и выполняем один forward pass. Если ячейка выводит loss, формат текста, изображений и LoRA совместимы с установленной версией InternVL.

In [ ]:
class InternVLCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer
        self.image_context_token = '<IMG_CONTEXT>'

    def prompt(self, question, image_tokens):
        return f'<img>{self.image_context_token * image_tokens}</img>\n{question}'

    def __call__(self, features):
        image_tokens = model.num_image_token
        prompts = [self.prompt(item['question'], image_tokens) for item in features]
        answers = [item['answer'] for item in features]
        full_texts = [f'{prompt}\n{answer}{tokenizer.eos_token}' for prompt, answer in zip(prompts, answers)]
        batch = self.tokenizer(full_texts, padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors='pt')
        prompt_batch = self.tokenizer(prompts, padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors='pt')
        labels = batch.input_ids.clone()
        labels[batch.attention_mask == 0] = -100
        for i, prompt_length in enumerate(prompt_batch.attention_mask.sum(dim=1).tolist()):
            labels[i, :prompt_length] = -100
        pixel_values = torch.stack([image_transform(load_image(item['image_path'])) for item in features])
        return {'input_ids': batch.input_ids, 'attention_mask': batch.attention_mask, 'labels': labels, 'pixel_values': pixel_values}

collator = InternVLCollator(tokenizer)
test_batch = collator([dataset[0]])
{name: tuple(value.shape) for name, value in test_batch.items()}

In [ ]:
device = next(model.parameters()).device
forward_batch = {name: value.to(device) for name, value in test_batch.items()}
with torch.no_grad():
    output = model(**forward_batch)
print(f'Forward pass выполнен. Loss на одном примере: {output.loss.item():.4f}')

## 5. Запуск A1

Эта ячейка подготовлена, но **не запускается в рамках проверки**. Перед запуском убедитесь, что F1 остановлен или завершён. Чекпойнты сохраняются раз в 50 оптимизационных шагов и не попадают в Git; для тяжёлых файлов будет использоваться DVC/DagsHub.

In [ ]:
training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR), num_train_epochs=1,
    per_device_train_batch_size=1, gradient_accumulation_steps=8,
    learning_rate=2e-4, lr_scheduler_type='cosine', warmup_steps=8,
    logging_steps=5, save_strategy='steps', save_steps=50, save_total_limit=2,
    fp16=True, gradient_checkpointing=True, optim='paged_adamw_8bit',
    report_to='none', remove_unused_columns=False, seed=SEED,
)
trainer = Trainer(model=model, args=training_args, train_dataset=dataset, data_collator=collator)
print(f'Примерное число оптимизационных шагов: {len(dataset) // training_args.gradient_accumulation_steps}')

# Запускайте вручную после завершения F1.
# trainer.train()
# model.save_pretrained(MODEL_DIR)
# tokenizer.save_pretrained(MODEL_DIR)
# (RESULTS_DIR / 'a1_internvl25_training_config.json').write_text(json.dumps({
#     'base_model': MODEL_ID, 'examples': len(dataset), 'lora_rank': 8,
#     'epochs': 1, 'learning_rate': 2e-4, 'seed': SEED,
# }, ensure_ascii=False, indent=2), encoding='utf-8')